# `C02` — The single crossing pair

Scenario **C02** of the Track A certification ladder (card `OMIE-CERT-C02-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C2). Oracle kind: **exact-parametric**. High-risk scenario (independence rule 4): its K4 audit is a genuinely separate agent.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality** in ticks (price tick 0.01 EUR/MWh, quantity tick 0.1 MW); euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** This certifies the single most consequential rule in the market specification: **a trade
executes at the resting order's limit price, never at the aggressor's**. If this is wrong, the engine
still trades, still conserves volume, still balances cash — every conservation invariant passes — and
every euro figure the thesis reports is wrong by the spread. There is no cheaper place to catch it
and no scenario that matters more. The draw is randomised over price, gap, both sizes and side so the
rule is pinned across the grid rather than at one lucky point, and the assertion is in integer ticks
so it cannot be blurred by floating-point decode.

**Certifies** (card): §6.1 ranking and match condition; §6.2 item 3 (price determination); §11.1 row *"Execution price = resting order's limit price in regular matching"*.

**How the claim is pinned.** One resting order, then one aggressor of the opposite side priced through it by $\Delta \ge 1$ tick. With $\Delta \ge 1$ the two limits differ, so a trade printed at the aggressor's limit cannot pass. Per draw the notebook asserts, in integer ticks: exactly one trade; its price is $p_{rest}$; its quantity is $\min(q_{rest}, q_{agg})$; and the engine's own trade row names the resting order as the passive (price-setting) leg and the aggressor as the aggressor, with buyer and seller on their declared sides.

**K2 boundary twins** (section 4). The generator draws $\Delta \in [1, 50]$ ticks, prices in [20, 120] EUR/MWh and sizes in [1, 60] MW. Equal sizes are drawn about once in 590 draws, so a 500-draw run misses them about 43 % of the time; the match boundary $\Delta = 0$ and the one-tick-short non-match are never drawn. Six deterministic twins of the first `TWIN_DRAWS` valid draws close these gaps, each changing **only the aggressor row**: one tick through, exactly at the touch ($\Delta = 0$, still a match under §6.1's $\ge$), one tick short (no match), equal sizes, a wide gap of 50 EUR/MWh, and both prices moved into negative territory.

**Does not certify.**
- Ranking among **several** resting orders, ties and time priority: C03. With one resting order §6.1's ranking can only pick that order; it is quoted because the card lists it, and it is exercised only in that trivial form.
- What the residual does after the trade (its timestamp and priority, §5.2, §6.2 item 4): C05. The residual's **quantity** is graded by the battery's I2 (conservation), not by a C02 oracle row.
- Multi-level walks (§6.2 item 6): C06. Iceberg price-setting (§6.2 item 7): C08 and C09. Batch matching (§6.3): out of scope for a single-zone simulator.
- Euro figures and the settlement cash leg: C12. They are computed from the same trade rows, so they cannot see a wrong price (I6's *Cannot see*).


## 2. The rule, quoted

The governing passages, quoted **verbatim** and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory source behind §6.2 item 3 is shown as a citation, not machine-checked.

How the quotes ground the expectation of section 5:

- **Who rests and who aggresses (§6.2 item 1, §4.5).** The first order meets an empty book: there is no counterpart, so it does not match, and a NON order that is not matched enters the book (§4.5, *Book entry if unmatched: Yes*). It is the **resting** order. The second order is the incoming order: it attempts to match against the best counterpart, which is the resting order, since it is the only one (§6.1's ranking, trivially).
- **Whether they match (§6.1, §6.2 item 2).** Opposite sides, same contract, and $\text{sell price} \le \text{buy price}$: equivalently, best bid $\ge$ best ask. Equality is a match (the touch twin); one tick short is not (the one-short twin).
- **At what price (§6.2 item 3, §11.1).** The trade executes at the **resting order's limit price**. §11.1 lists it as a hard rule from Continuous Trading Matching Algorithm §5.1.1.
- **For what quantity (derived; finding F1).** `market-spec.md` does not state $\min(q_{rest}, q_{agg})$ in words. It follows from §4.5 (NON orders can be partially filled), §5.1 (an accepted order ends `EXECUTED` or `PARTIALLY_EXECUTED + resting in BOOK`) and §6.2 item 4 (an incoming NON residual enters the book rather than trading again), together with the cited source's own statement that a matched order's quantity is *reduced by the trade quantity*. A trade cannot exceed either order's quantity, and matching stops only when one side is exhausted. With a single resting order the result is one trade of $\min(q_{rest}, q_{agg})$.


In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §6.1: ranking and the match condition -----------------------------------------
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Buy side:** ranked highest price first; ties broken by earliest timestamp.",
        note="Ranking (buy side). With one resting order it can only select that order; C03 certifies ranking among several.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Sell side:** ranked lowest price first; ties broken by earliest timestamp.",
        note="Ranking (sell side), same remark.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "A match occurs when best bid price ≥ best ask price.",
        note="The match condition. '≥': a bid equal to the ask matches (touch twin); one tick short does not (one-short twin).",
    ),
    # --- §6.2: regular matching ------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§6.2 item 1",
        "Incoming order attempts to match against best counterpart in the consolidated order book.",
        note="The second order is the incoming order; the first, already in the book, is its counterpart.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 2",
        "Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.",
        note="Every C02 draw is built to satisfy this (section 4 checks it from the tape).",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 3",
        "Trade executes at the **resting order's limit price** (maker pricing). "
        "Source: Continuous Trading Matching Algorithm §5.1.1.",
        note="THE rule of C02: the trade price is p_rest, never p_agg.",
    ),
    SpecQuote(
        "market-spec.md", "§11.1",
        "| Execution price = resting order's limit price in regular matching | "
        "Continuous Trading Matching Algorithm §5.1.1 |",
        note="§11.1 lists maker pricing as a hard rule, sourced to Matching Algorithm §5.1.1 (not to ACER/CACM, which source "
             "the price-time-priority row that C03 cites).",
    ),
    # --- who rests, and the quantity (derived; finding F1) ---------------------------------
    SpecQuote(
        "market-spec.md", "§4.5",
        "| Restriction | Partial fill | Book entry if unmatched | Applicable to |",
        note="Header of the execution-restriction table, quoted so the NON row below can be read.",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| NON | Yes | Yes | Regular, Block |",
        note="A NON order may be partially filled, and enters the book if unmatched: the first order rests.",
    ),
    SpecQuote(
        "market-spec.md", "§5.1",
        "├─ EXECUTED (fully matched immediately)",
        note="The incoming order when q_agg <= q_rest.",
    ),
    SpecQuote(
        "market-spec.md", "§5.1",
        "├─ PARTIALLY_EXECUTED + resting in BOOK",
        note="The incoming order when q_agg > q_rest: one trade of q_rest, the rest enters the book (not traded again).",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 4",
        "If incoming order is partially filled and has NON restriction: residual enters the book with its original "
        "entry timestamp (see §5.2).",
        note="Context only: the residual rests rather than trading again. Its timestamp and priority are C05's, not asserted here.",
    ),
)
CITATIONS = (
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.1.1 Price determination — "
        "docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/"
        "Continuous_Trading_Matching_Algorithm_Public_Description.md",
        "Two orders are matched at the limit price of the order that was already in the order book. If a buy order "
        "with a new timestamp is matched against an existing sell order, the limit price of the sell order becomes "
        "the trade execution price. If a sell order with a new timestamp is matched against an existing buy order, "
        "the limit price of the buy ord er becomes the trade execution price.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5 Order matching — same file",
        "When an order is matched in a tr ade, its quantity is reduced by the trade quantity. Orders with the "
        "execution restriction FOK or AON can only be matched with their full quantity. Orders with the execution "
        "restriction NON or IOC can also be matched partially.",
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"quotes: {sum(c.ok for c in QUOTE_CHECKS)}/{len(QUOTE_CHECKS)} match their source")


## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`**, the card's default for a random scenario. One draw is one seed of the generator's `C02.crossing_pair` construction: two orders, run jitted.
- **`SEED0 = 0`**, the first seed. A draw found outside the capacity margin before the run, or marked I8 `invalid` by the battery after it, is **discarded and replaced by the next seed**; the loop stops at `DRAWS` valid draws or after `2 × DRAWS` attempts, and the discard rate is reported.
- **`EAGER_DRAWS = 25`.** The first `EAGER_DRAWS` **valid** draws also run eager, and are compared integer field by integer field with their jitted run (`compare_records`). On top of these, the first valid draw of each of the four classes *(aggressor BUY or SELL)* × *(q_agg < q_rest or q_agg > q_rest)* not yet covered also runs eager, and so does the first run of each boundary twin. Every class and every twin variant **must** have one eager comparison: a missing one is recorded as a failed `eager_jit` assertion, never as an absent one. Because the eager runs are chosen among valid draws, a discarded seed cannot remove them. (An eager C02 run costs about 0.2 s after warm-up; jitted, about 0.01 s.)
- **`TWIN_DRAWS = 5`.** The first `TWIN_DRAWS` valid draws each also run the six boundary twins of section 4.
- The tick grids are **not** parameters: they come from `OMIEProductConfig()` and serve only to display euros. Every assertion is in integer ticks.

A run with far fewer draws than the default (say, under 20) can miss one of the four classes by chance; it then fails the coverage check and the missing class's eager comparison. That is intended: such a run does not cover the card's dataset.


In [ ]:
CERT_ID = "C02"      # C00-C16 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500          # valid draws of C02.crossing_pair (card: random scenarios default to 500 draws)
SEED0 = 0            # first seed
EAGER_DRAWS = 25     # the first EAGER_DRAWS valid draws also run eager (plus per-class and per-twin minimums)
TWIN_DRAWS = 5       # the first TWIN_DRAWS valid draws also run the six boundary twins


In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side

CONSTRUCTION = "C02.crossing_pair"
CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, ICE = int(ExecRestriction.NON), int(ExecRestriction.ICE)

SEEDS = list(range(SEED0, SEED0 + DRAWS))
MAX_ATTEMPTS = 2 * DRAWS      # the draw loop stops at DRAWS valid draws or after 2 x DRAWS attempts
SLICE_MARGIN = 64             # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
WIDE_GAP_TICKS = 5_000        # wide-gap twin: 50.00 EUR/MWh through the resting price
NEG_SHIFT_TICKS = 15_000      # negative-price twin: both limits moved down by 150.00 EUR/MWh
PRICE_INT_MAX = 1_999_800     # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C02_single_crossing_pair.ipynb"

print(f"{CERT_ID}: {CONSTRUCTION}, {DRAWS} valid draws from seed {SEED0} (at most {MAX_ATTEMPTS} attempts); "
      f"eager on the first {EAGER_DRAWS} valid draws + per-class minimums; twins on the first {TWIN_DRAWS}")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg")


## 4. Dataset

**The generator's draw.** `C02.crossing_pair` (`omie_tape_synth.make_crossing_pair`) draws, per seed: the resting side (BUY or SELL with probability ½), a resting limit $p_{rest}$ on the grid in [20, 120] EUR/MWh, a gap $\Delta \in [1, 50]$ ticks, and two sizes on the grid in [1, 60] MW. It emits two NON rows on one contract: **row 1, the resting order** (order id 1, agent 1, timestamp 1) at $p_{rest}$ for $q_{rest}$; **row 2, the aggressor** (order id 2, agent 2, timestamp 2) on the other side, priced $\Delta$ ticks **through** the resting price ($p_{agg} = p_{rest} + \Delta$ for a BUY aggressor, $p_{rest} - \Delta$ for a SELL aggressor), for $q_{agg}$. The rest of the tape is pad rows. Rows are submitted in tape order.

**What section 5 reads.** The expectation reads the **tape rows**: their order, sides, prices, quantities and order ids. `draw.params` is used only for display and for a consistency check (the params must agree with the tape). `draw.expected` is never read.

**The precondition, checked from the tape for every valid draw** (an aggregate dataset assertion in section 7): exactly two real rows; both NON; the same contract; opposite sides; prices within §2.3's bounds and quantities within its bounds, as integers; distinct order ids; the aggressor priced through by $\Delta \ge 1$ tick; and `draw.params` equal to the tape's integers, with `gap_ticks` equal to $\Delta$.

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings). Two NON orders sit far inside it: one resting order per side, at most two trade units, a hop bound of 4, no icebergs, and so no slice count (the 64-slices-per-iceberg row is re-checked from the tape and reported as 0). It is still checked **before** each run: a draw whose analytic bounds exceed the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 icebergs) or whose largest $\lceil qty/peak \rceil$ exceeds 64 is discarded and re-drawn, never run. After the run, the battery's I8 confirms the flags.

**The six boundary twins (K2 controls).** For each of the first `TWIN_DRAWS` valid draws, six copies of its tape are made, each changing **only the aggressor row** (row 2). Write $s = +1$ for a BUY aggressor and $-1$ for a SELL aggressor, so a limit $\delta$ ticks through the resting price is $p_{rest} + s\,\delta$:

| twin | aggressor price | aggressor size | what it pins |
|---|---|---|---|
| `one_tick` | $p_{rest} + s$ | $q_{agg}$ | the smallest gap at which maker pricing is observable |
| `touch` | $p_{rest}$ | $q_{agg}$ | §6.1's $\ge$: equal limits still match |
| `one_short` | $p_{rest} - s$ | $q_{agg}$ | §6.1's $\ge$ from the other side: no match, both rest |
| `equal_size` | $p_{agg}$ (as drawn) | $q_{rest}$ | $\min$ at its tie: both orders fully executed |
| `wide_gap` | $p_{rest} + 5000\,s$ (50.00 EUR/MWh) | $q_{agg}$ | a spread far larger than the drawn ones |
| `negative` | $p_{agg} - 15000$, **and** the resting price $p_{rest} - 15000$ | $q_{agg}$ | the rule below zero EUR/MWh (resting order moved too) |

The `negative` twin is the one exception to "only row 2": it moves both limits down by 150.00 EUR/MWh, so $p_{rest}$ lands in [−130, −30] EUR/MWh with the drawn gap unchanged. Every twin stays within §2.3's bounds (checked in section 7 with the precondition).

Below: the first seed's rows in readable units beside the integers, a per-seed summary, the first seed's twins, and plots of the book shape and of the dataset's spread.


In [ ]:
from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

C_SIDE, C_PRICE, C_QTY, C_OID = synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY, synth.COL_OID
C_AGENT, C_TS, C_EXEC, C_CONTRACT, C_PEAK = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC,
                                             synth.COL_CONTRACT, synth.COL_PEAK)


def real_rows(tape):
    """The real rows (order_id >= 1), in tape (= submission) order."""
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def through(agg_side, ticks):
    """Signed offset of a limit `ticks` through the resting price for an aggressor on `agg_side`."""
    return int(ticks) if int(agg_side) == BUY else -int(ticks)


def max_slices(tape):
    """Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none)."""
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((-(-int(r[C_QTY]) // max(1, int(r[C_PEAK]))) for r in ice), default=0)


def precondition(draw):
    """Every way a draw departs from the dataset described above (empty list = as described)."""
    rows, p, bad = real_rows(draw.tape), draw.params, []
    if len(rows) != 2:
        return [f"{len(rows)} real rows (expected 2)"]
    r, a = rows
    if int(r[C_EXEC]) != NON or int(a[C_EXEC]) != NON:
        bad.append("an order is not NON")
    if int(r[C_CONTRACT]) != int(a[C_CONTRACT]):
        bad.append("different contracts")
    if int(r[C_SIDE]) == int(a[C_SIDE]):
        bad.append("same side")
    for x in (r, a):
        if not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(x[C_QTY]) <= QTY_INT_MAX:
            bad.append(f"oid {int(x[C_OID])} outside §2.3's bounds")
    if int(r[C_OID]) == int(a[C_OID]):
        bad.append("duplicate order id")
    gap = (int(a[C_PRICE]) - int(r[C_PRICE])) * through(a[C_SIDE], 1)
    if gap < 1:
        bad.append(f"aggressor not priced through: gap {gap} ticks")
    want = {"rest_side": int(r[C_SIDE]), "agg_side": int(a[C_SIDE]), "rest_oid": int(r[C_OID]),
            "agg_oid": int(a[C_OID]), "p_rest_int": int(r[C_PRICE]), "p_agg_int": int(a[C_PRICE]),
            "q_rest_int": int(r[C_QTY]), "q_agg_int": int(a[C_QTY]), "gap_ticks": gap}
    diff = {k: (p.get(k), v) for k, v in want.items() if p.get(k) != v}
    if diff:
        bad.append(f"params disagree with the tape (params, tape): {diff}")
    return bad


def twin_tapes(draw):
    """The six boundary twins of section 4's table: only the aggressor row changes (and, for
    `negative`, the resting price too)."""
    base = np.array(draw.tape, copy=True)
    idx = np.nonzero(base[:, C_OID] >= 1)[0]
    i_r, i_a = int(idx[0]), int(idx[1])
    p_rest, p_agg = int(base[i_r, C_PRICE]), int(base[i_a, C_PRICE])
    q_rest = int(base[i_r, C_QTY])
    s_agg = int(base[i_a, C_SIDE])
    out = {}

    def make(name, *, agg_price=None, agg_qty=None, shift=0):
        t = base.copy()
        if agg_price is not None:
            t[i_a, C_PRICE] = agg_price
        if agg_qty is not None:
            t[i_a, C_QTY] = agg_qty
        if shift:
            t[i_r, C_PRICE] += shift
            t[i_a, C_PRICE] += shift
        out[name] = np.ascontiguousarray(t, dtype=np.int32)

    make("one_tick", agg_price=p_rest + through(s_agg, 1))
    make("touch", agg_price=p_rest)
    make("one_short", agg_price=p_rest - through(s_agg, 1))
    make("equal_size", agg_qty=q_rest)
    make("wide_gap", agg_price=p_rest + through(s_agg, WIDE_GAP_TICKS))
    make("negative", shift=-NEG_SHIFT_TICKS)
    return out


TWIN_VARIANTS = ("one_tick", "touch", "one_short", "equal_size", "wide_gap", "negative")
DRAWN, TWINS = {}, {}


def prepare(seed):
    """Draw seed `seed` and derive its twins (used for the initial seeds and for re-draws)."""
    d = synth.CONSTRUCTIONS[CONSTRUCTION](seed)
    DRAWN[seed], TWINS[seed] = d, twin_tapes(d)
    return d


for s in SEEDS:
    prepare(s)


def rows_table(tape, title):
    lines = [f"**{title}**", "",
             "| row | oid | side | price (EUR/MWh) | qty (MW) | price_int | qty_int | agent | ts | exec |",
             "|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in enumerate(real_rows(tape), start=1):
        lines.append(
            f"| {k} ({'resting' if k == 1 else 'aggressor'}) | {int(r[C_OID])} | {side_name(r[C_SIDE])} | "
            f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | "
            f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_AGENT])} | {int(r[C_TS])} | "
            f"{'NON' if int(r[C_EXEC]) == NON else int(r[C_EXEC])} |")
    return "\n".join(lines)


first = DRAWN[SEED0]
display(Markdown(rows_table(first.tape, f"Seed {SEED0} (digest `{first.digest()[:16]}…`, tape {first.tape.shape}, "
                                        f"hop bound {first.hop_bound})")))
for s in SEEDS[:8]:
    p = DRAWN[s].params
    print(f"seed {s}: resting {side_name(p['rest_side'])} {p['p_rest_eur']:.2f} EUR/MWh x {p['q_rest_mw']:.1f} MW "
          f"(int {p['p_rest_int']} / {p['q_rest_int']}); aggressor {side_name(p['agg_side'])} "
          f"{p['p_agg_eur']:.2f} x {p['q_agg_mw']:.1f} (int {p['p_agg_int']} / {p['q_agg_int']}); "
          f"gap {p['gap_ticks']} ticks")
_gaps = np.array([DRAWN[s].params["gap_ticks"] for s in SEEDS])
_cls = [(DRAWN[s].params["agg_side"], np.sign(DRAWN[s].params["q_agg_int"] - DRAWN[s].params["q_rest_int"]))
        for s in SEEDS]
print(f"... {len(SEEDS)} seeds: aggressor BUY {sum(c[0] == BUY for c in _cls)}, SELL {sum(c[0] == SELL for c in _cls)}; "
      f"q_agg < q_rest {sum(c[1] < 0 for c in _cls)}, = {sum(c[1] == 0 for c in _cls)}, > {sum(c[1] > 0 for c in _cls)}; "
      f"gap 1 tick {int(np.sum(_gaps == 1))}, gap range [{_gaps.min()}, {_gaps.max()}] ticks")
print(f"largest analytic bounds over the seeds: "
      + ", ".join(f"{k} {max(DRAWN[s].capacity_bounds[k] for s in SEEDS)}" for k in synth.AXES)
      + f"; largest ceil(qty/peak) {max(max_slices(DRAWN[s].tape) for s in SEEDS)}")

for name, t in TWINS[SEED0].items():
    display(Markdown(rows_table(t, f"Seed {SEED0}, twin `{name}`")))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
ax = axes[0]
for k, r in enumerate(real_rows(first.tape)):
    buy = int(r[C_SIDE]) == BUY
    ax.barh(decode_price(int(r[C_PRICE]), CONFIG), decode_qty(int(r[C_QTY]), CONFIG) * (1 if buy else -1),
            height=0.02, color="tab:green" if buy else "tab:red",
            label=f"{'resting' if k == 0 else 'aggressor'} {side_name(r[C_SIDE])}")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("quantity (MW): buy right, sell left")
ax.set_ylabel("limit price (EUR/MWh)")
ax.set_title(f"seed {SEED0}: the two orders")
ax.legend(loc="best", fontsize=7)

axes[1].hist(_gaps, bins=np.arange(0.5, 51.5, 1.0), color="tab:blue")
axes[1].set_xlabel("gap Δ (ticks of 0.01 EUR/MWh)")
axes[1].set_ylabel("draws")
axes[1].set_title("aggressor priced through by Δ")

q_r = [DRAWN[s].params["q_rest_mw"] for s in SEEDS]
q_a = [DRAWN[s].params["q_agg_mw"] for s in SEEDS]
col = ["tab:green" if DRAWN[s].params["agg_side"] == BUY else "tab:red" for s in SEEDS]
axes[2].scatter(q_r, q_a, s=5, c=col, alpha=0.6)
axes[2].plot([0, 60], [0, 60], color="black", lw=0.8)
axes[2].set_xlabel("q_rest (MW)")
axes[2].set_ylabel("q_agg (MW)")
axes[2].set_title("sizes (green: BUY aggressor; diagonal: equal)")
plt.tight_layout()
plt.show()


## 5. Known result

**Notation.** Integer ticks throughout: a price $p$ is its offset-encoded tick index, a quantity $q$ its count of 0.1 MW. The tape's two real rows, in submission order, are $r$ (first) and $a$ (second), with sides $\sigma_r, \sigma_a$, limits $p_r, p_a$, quantities $q_r, q_a$ and order ids $o_r, o_a$. Both are NON, on one contract, on opposite sides (section 4's precondition). Let $p_{buy}$ and $p_{sell}$ be the limits of the BUY row and the SELL row.

**(M) Match condition** (§6.1, §6.2 item 2):
$$\text{match} \iff p_{sell} \le p_{buy}.$$
The first row meets an empty book and rests (§4.5; §6.2 item 1 finds no counterpart). The second row is the incoming order, and its only counterpart is $r$.

**(T) If they match: exactly one trade** (§6.2 items 1 and 3; quantity derived, finding F1):
$$n_{trades} = 1,\qquad \text{price} = p_r,\qquad \text{qty} = \min(q_r, q_a).$$
The trade row names $r$ as the **passive** leg (the price-setter) and $a$ as the **aggressor**. Its buyer is the BUY row's order id and its seller the SELL row's. It is written by $a$'s submission, and $r$'s own submission writes no trade.

**(N) If they do not match** (the `one_short` twin only): $n_{trades} = 0$, and both orders rest at their own limits with their full quantities (§4.5, *Book entry if unmatched: Yes*), each on its own side.

For a generator draw, $r$ is the resting order and the gap is $\Delta \ge 1$, so (T) applies with $p_r = p_{rest} \ne p_{agg}$. The price assertion therefore **separates** maker pricing from taker pricing on every draw: a trade at $p_{agg}$ misses by $\Delta$ ticks.

**What each twin gives, by construction** ($p_{rest}, q_{rest}, p_{agg}, q_{agg}$ are the base draw's; $s = \pm 1$ as in section 4):

| twin | match? | price | qty |
|---|---|---|---|
| `one_tick` | yes | $p_{rest}$ (aggressor at $p_{rest} + s$) | $\min(q_{rest}, q_{agg})$ |
| `touch` | yes (equality) | $p_{rest} = p_{agg}$ | $\min(q_{rest}, q_{agg})$ |
| `one_short` | **no** | — (both rest, full size) | 0 |
| `equal_size` | yes | $p_{rest}$ | $q_{rest}$ (nothing rests afterwards) |
| `wide_gap` | yes | $p_{rest}$ (aggressor 5,000 ticks through) | $\min(q_{rest}, q_{agg})$ |
| `negative` | yes | $p_{rest} - 15{,}000$ | $\min(q_{rest}, q_{agg})$ |

The code below implements (M), (T) and (N) as **one function of the tape rows**, used unchanged for draws and twins. It does not read `draw.params` or `draw.expected`, and it does not run the engine. The table above is checked against it on the first seed's twins.


In [ ]:
def expectation(tape):
    """(M), (T), (N) from the tape's two real rows, in integer ticks. None if the tape is not a pair."""
    rows = real_rows(tape)
    if len(rows) != 2:
        return None
    r, a = rows
    buy, sell = (r, a) if int(r[C_SIDE]) == BUY else (a, r)
    crosses = (int(r[C_SIDE]) != int(a[C_SIDE]) and int(r[C_CONTRACT]) == int(a[C_CONTRACT])
               and int(sell[C_PRICE]) <= int(buy[C_PRICE]))
    e = {
        "crosses": bool(crosses),
        "rest_oid": int(r[C_OID]), "agg_oid": int(a[C_OID]),
        "rest_side": int(r[C_SIDE]), "agg_side": int(a[C_SIDE]),
        "p_rest_int": int(r[C_PRICE]), "p_agg_int": int(a[C_PRICE]),
        "q_rest_int": int(r[C_QTY]), "q_agg_int": int(a[C_QTY]),
    }
    if crosses:
        e.update(n_trades=1, trade_price_int=int(r[C_PRICE]), trade_qty_int=min(int(r[C_QTY]), int(a[C_QTY])),
                 who={"passive_oid": int(r[C_OID]), "aggressor_oid": int(a[C_OID]),
                      "buyer_oid": int(buy[C_OID]), "seller_oid": int(sell[C_OID]),
                      "on_aggressor_submission": True})
    else:
        e.update(n_trades=0, trade_price_int=None, trade_qty_int=0,
                 resting={side_name(r[C_SIDE]): [[int(r[C_PRICE]), int(r[C_QTY]), int(r[C_OID])]],
                          side_name(a[C_SIDE]): [[int(a[C_PRICE]), int(a[C_QTY]), int(a[C_OID])]]})
    return e


def twin_table_expectation(draw, variant):
    """Section 5's twin table, written out per row (checked against `expectation` below)."""
    p = draw.params
    m = min(p["q_rest_int"], p["q_agg_int"])
    return {
        "one_tick": (1, p["p_rest_int"], m),
        "touch": (1, p["p_rest_int"], m),
        "one_short": (0, None, 0),
        "equal_size": (1, p["p_rest_int"], p["q_rest_int"]),
        "wide_gap": (1, p["p_rest_int"], m),
        "negative": (1, p["p_rest_int"] - NEG_SHIFT_TICKS, m),
    }[variant]


EXPECTED = {s: expectation(DRAWN[s].tape) for s in SEEDS}
for s in SEEDS[:8]:
    e = EXPECTED[s]
    print(f"seed {s}: {e['n_trades']} trade at price_int {e['trade_price_int']} "
          f"({decode_price(e['trade_price_int'], CONFIG):.2f} EUR/MWh; aggressor limit {e['p_agg_int']}, "
          f"{abs(e['p_agg_int'] - e['p_rest_int'])} ticks away) for qty_int {e['trade_qty_int']} "
          f"({decode_qty(e['trade_qty_int'], CONFIG):.1f} MW); passive oid {e['who']['passive_oid']}, "
          f"aggressor oid {e['who']['aggressor_oid']}")

TWIN_TABLE_AGREES = True
for v, t in TWINS[SEED0].items():
    e = expectation(t)
    got = (e["n_trades"], e["trade_price_int"], e["trade_qty_int"])
    want = twin_table_expectation(DRAWN[SEED0], v)
    TWIN_TABLE_AGREES &= got == want
    print(f"seed {SEED0} twin {v:<10}: expectation {got}  table {want}  {'agree' if got == want else 'DISAGREE'}")


## 6. Run

### Eager and jitted (wired at K3)

> **K3: this section is yours; sections 1–5 and 7 are locked (independence rule 2).** A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly **one name** from this section. Until it is callable, section 7 records a failing `section 6 wired` check and the notebook ends `NOT CERTIFIED`.

**`run_tape(tape, *, jit) -> RunRecord`**: the battery's record builder on one tape, i.e. `tests.omiecid.reference.omie_invariants.build_record(tape, jit=jit)`, on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), no scheduled cancellations, no gate-closure expiry, and the default capacity bounds (`tape_capacity_bounds(tape)`). `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. The tape is passed **as is**: a generator draw's `DRAWN[seed].tape`, or one of its twins `TWINS[seed][variant]` (section 4). Nothing in the tape may be rewritten.

Section 7 reads these `RunRecord` fields: `fills` (each `Fill`'s `price_int`, `qty_int`, `passive_oid`, `aggressor_oid`, `buyer_oid`, `seller_oid`, `submission`), `accepted` (its keys and each `AcceptedOrder.submission`), `final_snapshot.resting(side)`, `observed_capacity`, and the whole record through `run_battery` and `compare_records`.

Section 7 calls `run_tape(..., jit=True)` on every draw and every twin it runs, and `run_tape(..., jit=False)` on the eager selection of section 3. A seed beyond the initial range (a re-draw after a discard) is prepared with `prepare(seed)` before the call.


In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_tape(tape, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one fresh book of one OPEN contract, with the default
# configs, no scheduled cancellations, no gate-closure expiry and the default analytic
# capacity bounds (tape_capacity_bounds(tape)). jit=False wraps the whole run in
# jax.disable_jit() (inside build_record); jit=True runs it jitted. The tape is passed as is.
# ---------------------------------------------------------------------------------------
from omiecid_simulator.omiecid.contract import ContractStatus
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig
from tests.omiecid.reference.omie_invariants import build_record

MARKET = OMIEMarketConfig()


def run_tape(tape, *, jit):
    """One run of `tape` (a draw's tape or one of its twins, unmodified) on a fresh OPEN book."""
    return build_record(tape, product_config=CONFIG, market_config=MARKET,
                        contract_status=ContractStatus.OPEN, cancels_before={},
                        expire_at_gct=False, capacity_bounds=None, jit=jit)


print(f"run_tape: build_record on one OPEN contract, {type(CONFIG).__name__}() + {type(MARKET).__name__}(), "
      f"no cancels, no GCT expiry, default capacity bounds")

## 7. Certification

The oracle comparison in integer ticks and order ids comes first, then the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid draw** (so an auditor can check the exact integers from the results JSON alone, not only a folded flag), four assertions, each with its expected and observed value:
1. `seed s: number of trades`: expected 1 (`count`);
2. `seed s: trade price (resting order's limit)`: expected $p_{rest}$; observed the one trade's `price_int`, or the list of all trade prices if there is not exactly one (`price_int`);
3. `seed s: trade quantity`: expected $\min(q_{rest}, q_{agg})$; observed the sum of the trade quantities (`qty_int`);
4. `seed s: who traded`: expected and observed `{passive_oid, aggressor_oid, buyer_oid, seller_oid, on_aggressor_submission}` for every trade row, plus the accepted order ids (`other`).

Twin runs record the same four, named `seed s twin <variant>: …`, plus one assertion that their battery passed (a twin is not a draw, so it does not enter the draw accounting or the invariant table). The `one_short` twin records the resting book in place of the price and the trade identities.

**Aggregate assertions:** the section-6 wiring; the dataset precondition (section 4) over every valid draw, listing failing seeds; the twin table of section 5 against the expectation function; the coverage of the four classes; and the eager comparisons, one per required class and twin variant, a missing one recorded as a failure.

The battery runs on each draw's jitted record with `expects_trade=True`. A draw the battery marks I8 `invalid` is discarded and replaced by the next seed. A per-draw table (all failing draws, then the first 20) is shown before the full certificate table.


In [ ]:
import traceback
from collections import Counter

from tests.omiecid.reference.omie_invariants import compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="The single crossing pair",
    known_result="exactly one trade; price_int = p_rest (the resting order's limit, never the aggressor's); "
                 "qty_int = min(q_rest, q_agg); passive leg = the resting order",
    oracle_kind="exact-parametric", quotes=QUOTES, citations=CITATIONS, constructions=(CONSTRUCTION,),
    requested_draws=DRAWS, runtime_forecast_s=60.0, started_at=T0,
    downstream="every PnL and euro figure (price attribution); invariant I5; C05 (residual after a trade), "
               "C06 (each level of a walk priced at its maker), C08-C10 (iceberg as price-setter), "
               "C12 (settlement computed from these trade prices); Track B rewards",
)

R_MATCH = "§6.1; §6.2 item 2"
R_PRICE = "§6.2 item 3; §11.1"
R_QTY = "§4.5; §5.1; §6.2 items 1, 4 (qty = min, derived: finding F1)"
R_WHO = "§6.2 items 1, 3 (the resting order is the price-setter)"
R_REST = "§4.5 (NON: book entry if unmatched)"
R_DATA = "§6.2 item 2; §2.3 (dataset precondition, section 4)"
STRICT_CLASSES = (("BUY", "q_agg<q_rest"), ("BUY", "q_agg>q_rest"), ("SELL", "q_agg<q_rest"), ("SELL", "q_agg>q_rest"))


def size_class(e):
    d = e["q_agg_int"] - e["q_rest_int"]
    return (side_name(e["agg_side"]), "q_agg<q_rest" if d < 0 else "q_agg>q_rest" if d > 0 else "q_agg=q_rest")


def guarded(label, seed, fn, *args, **kw):
    """Call a section-6 function; a raise is recorded as a failed check, never propagated."""
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_PRICE)
        return None


def observe(rec, e):
    """What the engine did, read from the record, in the expectation's terms."""
    fills = list(rec.fills)
    agg_sub = rec.accepted[e["agg_oid"]].submission if e["agg_oid"] in rec.accepted else None
    who = [{"passive_oid": int(f.passive_oid), "aggressor_oid": int(f.aggressor_oid),
            "buyer_oid": int(f.buyer_oid), "seller_oid": int(f.seller_oid),
            "on_aggressor_submission": agg_sub is not None and int(f.submission) == int(agg_sub)}
           for f in fills]
    snap = rec.final_snapshot
    return {
        "n_trades": len(fills),
        "price": int(fills[0].price_int) if len(fills) == 1 else [int(f.price_int) for f in fills],
        "qty": int(sum(int(f.qty_int) for f in fills)),
        "who": who,
        "accepted": sorted(int(k) for k in rec.accepted),
        "resting": {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in snap.resting(sd))
                    for sd in (BUY, SELL)},
    }


def check_run(label, seed, rec, e):
    """The four per-run assertions of (T), or (N) for a non-crossing tape. Returns the observation."""
    o = observe(rec, e)
    acc_want = sorted([e["rest_oid"], e["agg_oid"]])
    cert.check_equal(f"{label}: number of trades", o["n_trades"], e["n_trades"], seed=seed, unit="count",
                     rule=R_MATCH)
    if e["crosses"]:
        cert.check_equal(f"{label}: trade price (resting order's limit)", o["price"], e["trade_price_int"],
                         seed=seed, unit="price_int", rule=R_PRICE,
                         detail=f"p_rest {e['p_rest_int']}, p_agg {e['p_agg_int']}")
        cert.check_equal(f"{label}: trade quantity", o["qty"], e["trade_qty_int"], seed=seed, unit="qty_int",
                         rule=R_QTY, detail=f"q_rest {e['q_rest_int']}, q_agg {e['q_agg_int']}")
        cert.check_equal(f"{label}: who traded", {"trades": o["who"], "accepted": o["accepted"]},
                         {"trades": [e["who"]], "accepted": acc_want}, seed=seed, unit="other", rule=R_WHO)
    else:
        cert.check_equal(f"{label}: trade quantity", o["qty"], 0, seed=seed, unit="qty_int", rule=R_MATCH)
        want_rest = {k: sorted(v) for k, v in e["resting"].items()}
        cert.check_equal(f"{label}: both orders rest unchanged", {"resting": o["resting"], "accepted": o["accepted"]},
                         {"resting": want_rest, "accepted": acc_want}, seed=seed, unit="other", rule=R_REST)
    return o


def ok_row(o, e):
    if e["crosses"]:
        return (o["n_trades"] == 1 and o["price"] == e["trade_price_int"] and o["qty"] == e["trade_qty_int"]
                and o["who"] == [e["who"]])
    return o["n_trades"] == 0 and o["qty"] == 0


def eager_compare(label, seed, tape, rec):
    """Run `tape` eager and compare with its jitted record; recorded as one eager_jit assertion."""
    rec_e = guarded(f"{label} (eager)", seed, run_tape, tape, jit=False)
    if rec is None or rec_e is None:
        diff = ["jitted run missing" if rec is None else "eager run missing"]
    else:
        diff = list(compare_records(rec_e, rec))
    cert.set_eager_jit(seed, diff == [], detail=f"{label}: " + (", ".join(diff) if diff else "identical"))
    return diff == []


def pre_run_margin(draw):
    """The capacity-validity precondition, checked from the draw before it is run."""
    cb, bad = dict(draw.capacity_bounds), []
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    for k, m in lim.items():
        if int(cb.get(k, 0)) > m:
            bad.append(f"{k} bound {cb.get(k)} > {m}")
    if max_slices(draw.tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(draw.tape)} > {SLICE_MARGIN}")
    return bad


# ---- the expectation's own twin table agrees with the expectation function ------------------
cert.check("section 5: twin table agrees with the expectation function (seed SEED0)", TWIN_TABLE_AGREES,
           expected=True, observed=TWIN_TABLE_AGREES, unit="bool", rule=R_PRICE)

wired = callable(run_tape)
cert.check("section 6 wired (run_tape defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_PRICE)

PER_DRAW, DATA_BAD, CLASS_COUNT = [], {}, Counter()
EAGER_CLASS, EAGER_TWIN, TWIN_SEEDS = {}, {}, []
PEAK = {"hop_bound": 0, "max_slices": 0}
if wired:
    valid, seed = 0, SEED0
    while valid < DRAWS and seed < SEED0 + MAX_ATTEMPTS:
        draw = DRAWN.get(seed) or prepare(seed)
        margin = pre_run_margin(draw)
        if margin:
            cert.discard(seed, "pre-run capacity margin: " + "; ".join(margin))
            seed += 1
            continue
        e = EXPECTED.get(seed) or expectation(draw.tape)
        EXPECTED[seed] = e
        rec = guarded(f"seed {seed} (jit)", seed, run_tape, draw.tape, jit=True)
        if rec is not None and not cert.add_battery(run_battery(rec, expects_trade=True), seed=seed):
            seed += 1   # I8 invalid: discarded, re-drawn with the next seed
            continue
        valid += 1
        cert.record_case(seed, draw.tape, draw.params)
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        bad = precondition(draw)
        if bad:
            DATA_BAD[seed] = bad
        cls = size_class(e) if e is not None else None
        CLASS_COUNT[cls] += 1
        if rec is None or e is None:
            if e is None:
                cert.check(f"seed {seed}: expectation defined", False, seed=seed, expected="a two-row tape",
                           observed=len(real_rows(draw.tape)), unit="count", rule=R_DATA)
            seed += 1
            continue
        cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
        o = check_run(f"seed {seed}", seed, rec, e)
        PER_DRAW.append({"seed": seed, "e": e, "o": o, "ok": ok_row(o, e)})

        # ---- eager vs jitted: on valid draws only, by count and by class --------------------
        if valid <= EAGER_DRAWS or cls not in EAGER_CLASS:
            eager_compare(f"seed {seed}", seed, draw.tape, rec)   # a failed comparison is recorded; it still counts as run
            EAGER_CLASS.setdefault(cls, seed)

        # ---- the boundary twins, on the first TWIN_DRAWS valid draws ------------------------
        if len(TWIN_SEEDS) < TWIN_DRAWS:
            TWIN_SEEDS.append(seed)
            for v in TWIN_VARIANTS:
                t = TWINS[seed][v]
                lbl = f"seed {seed} twin {v}"
                et = expectation(t)
                rt = guarded(f"{lbl} (jit)", seed, run_tape, t, jit=True)
                if rt is None:
                    continue
                rep = run_battery(rt, expects_trade=et["crosses"])
                cert.check(f"{lbl}: battery (RB, I1-I9) passes", rep.passed and rep.valid, seed=seed,
                           expected="pass / not_applicable", observed=rep.statuses(), unit="other", rule=R_PRICE)
                ot = check_run(lbl, seed, rt, et)
                PER_DRAW.append({"seed": seed, "twin": v, "e": et, "o": ot, "ok": ok_row(ot, et)})
                if v not in EAGER_TWIN:
                    eager_compare(lbl, seed, t, rt)
                    EAGER_TWIN[v] = seed
        seed += 1

# ---- aggregate assertions ------------------------------------------------------------------------
n_valid = sum(1 for r in PER_DRAW if "twin" not in r)
cert.check("dataset: every valid draw is the crossing pair of section 4 (precondition)", wired and not DATA_BAD,
           expected=[], observed={str(k): v for k, v in list(DATA_BAD.items())[:10]}, unit="other",
           detail=(f"{len(DATA_BAD)} draws depart; first: {list(DATA_BAD)[:10]}" if DATA_BAD
                   else f"{n_valid}/{n_valid} draws"), rule=R_DATA)
_tw_out = [f"seed {s} {v}" for s in TWIN_SEEDS for v in TWIN_VARIANTS
           for r in real_rows(TWINS[s][v])
           if not 0 <= int(r[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(r[C_QTY]) <= QTY_INT_MAX]
cert.check("dataset: every twin run is within §2.3's bounds", wired and not _tw_out, expected=[],
           observed=_tw_out[:10], unit="other", rule="§2.3")
missing_cls = [c for c in STRICT_CLASSES if CLASS_COUNT.get(c, 0) == 0]
cert.check("dataset: the four classes (aggressor side x size order) are all drawn", wired and not missing_cls,
           expected={"/".join(c): ">= 1" for c in STRICT_CLASSES},
           observed={"/".join(c) if c else "None": n for c, n in CLASS_COUNT.items()}, unit="count",
           detail=f"missing: {missing_cls}" if missing_cls else "", rule=R_DATA)
for c in STRICT_CLASSES:
    if c not in EAGER_CLASS:
        cert.set_eager_jit(None, False, detail=f"MISSING: no valid draw of class {'/'.join(c)} was compared eager")
for v in TWIN_VARIANTS:
    if v not in EAGER_TWIN:
        cert.set_eager_jit(None, False, detail=f"MISSING: twin {v} was never compared eager")

# ---- the per-draw table: failing rows first, then the first 20 --------------------------------
def _fmt(r):
    e, o = r["e"], r["o"]
    w = o["who"][0] if len(o["who"]) == 1 else o["who"]
    return (f"| {r['seed']} | {r.get('twin', '')} | {side_name(e['agg_side'])} | {e['p_rest_int']} | {e['p_agg_int']} | "
            f"{e['q_rest_int']} | {e['q_agg_int']} | {e['n_trades']} / {o['n_trades']} | "
            f"{e['trade_price_int']} / {o['price']} | {e['trade_qty_int']} / {o['qty']} | {w} | "
            f"{'PASS' if r['ok'] else 'FAIL'} |")


_hdr = ["| seed | twin | aggressor | p_rest | p_agg | q_rest | q_agg | trades exp / obs | price exp / obs | "
        "qty exp / obs | observed trade identities | row |", "|" + "---|" * 12]
_fail = [r for r in PER_DRAW if not r["ok"]]
display(Markdown(f"**Per-draw integers** ({len(PER_DRAW)} runs: {n_valid} draws + "
                 f"{len(PER_DRAW) - n_valid} twin runs; {len(_fail)} failing)\n\n"
                 + "\n".join(_hdr + [_fmt(r) for r in _fail] + [_fmt(r) for r in PER_DRAW[:20] if r["ok"]])))
print(f"classes among valid draws: {dict(CLASS_COUNT)}")
print(f"eager comparisons: classes {EAGER_CLASS}; twins {EAGER_TWIN}; "
      f"{sum(1 for a in cert.payload()['assertions'] if a['kind'] == 'eager_jit')} eager_jit assertions in all")
print(f"pre-run margins: largest hop bound {PEAK['hop_bound']} (<= {ENVELOPE.max_hops}), "
      f"largest ceil(qty/peak) {PEAK['max_slices']} (<= {SLICE_MARGIN})")

display(Markdown(cert.table()))


## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")


In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))
